In [31]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph

# Vector Stores
!pip install chromadb
!pip install langchain-chroma

# Import the Python SDK for Colab secrets
from google.colab import userdata


In [32]:
import json
import os
import numpy as np
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [33]:
# ============================================================================
# PART 1: Setup - Load Data and Build RAG System
# ============================================================================
print("\n" + "="*80)
print("PART 1: Setup RAG System for Evaluation")
print("="*80)



PART 1: Setup RAG System for Evaluation


In [37]:
# Load tickets
with open('./sample_data/synthetic_tickets.json', 'r', encoding='utf-8') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

# Load evaluation queries with ground truth
with open('./sample_data/evaluation_queries.json', 'r', encoding='utf-8') as f:
    eval_queries = json.load(f)
print(f"✓ Loaded {len(eval_queries)} evaluation queries with ground truth")

✓ Loaded 22 support tickets
✓ Loaded 15 evaluation queries with ground truth


In [38]:
# Build documents
documents = []
for ticket in tickets:
    content = f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}"""

    doc = Document(
        page_content=content,
        metadata={
            'ticket_id': ticket['ticket_id'],
            'title': ticket['title'],
            'category': ticket['category']
        }
    )
    documents.append(doc)

In [ ]:
# Initialize embeddings and LLM with timeout
embeddings = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'),
    request_timeout=30
)


In [39]:
# Use direct OpenAI client instead of LangChain ChatOpenAI (fixes connection pooling issue)
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'), timeout=30.0, max_retries=2)
chat_model = os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')

print("✓ OpenAI models initialized (30s timeout)")


# Test API connection
print("\nTesting OpenAI API connection...")
try:
    test_response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": "Say 'OK'"}],
        max_tokens=5
    )
    print("✓ API connection successful")
except Exception as e:
    print(f"✗ API connection failed: {str(e)[:100]}")
    print("\nPlease check:")
    print("  1. OPENAI_API_KEY is set correctly in .env")
    print("  2. You have internet connectivity")
    print("  3. Your API key is valid and has credits")
    exit(1)

✓ OpenAI models initialized (30s timeout)

Testing OpenAI API connection...
✓ API connection successful


In [40]:
# Build vector store
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="eval_supportdesk"
)
print("✓ Vector store built with Chroma")

✓ Vector store built with Chroma


In [41]:
# Create simple RAG function
def generate_answer(query, k=3):
    """
    Retrieve top-k supporting documents and generate a grounded answer.

    This helper centralizes the "retrieve + generate" path used by both
    retrieval and generation evaluations, ensuring consistent behavior.
    """
    # Retrieve documents
    docs = vector_store.similarity_search(query, k=k)

    # Build context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Create prompt
    prompt = f"""You are a technical support assistant. Answer the question using ONLY the provided context.

Context:
{context}

Question: {query}

Answer (cite ticket IDs):"""

    # Generate answer using direct OpenAI client
    response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    answer = response.choices[0].message.content

    return {'answer': answer, 'source_documents': docs}

print("✓ RAG function ready")

✓ RAG function ready


In [ ]:
# ============================================================================
# PART 2: RETRIEVAL LAYER EVALUATION
# ============================================================================
print("\n" + "="*80)
print("PART 2: RETRIEVAL LAYER EVALUATION")
print("="*80)
print("\nMetrics: Precision, Recall, F1 Score")
print("Measures: Are we retrieving the right documents?")

In [42]:
def calculate_retrieval_metrics(retrieved_ids, relevant_ids, k=3):
    """
    Calculate Precision, Recall, and F1 at k

    Precision@k = (relevant docs in top-k) / k
    Recall@k = (relevant docs in top-k) / (total relevant docs)
    F1@k = harmonic mean of Precision and Recall
    """
    # Restrict to top-k because retrieval quality is rank-sensitive.
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)

    # True positives: relevant docs that were retrieved
    tp = len(retrieved_set & relevant_set)

    # Precision: what % of retrieved docs are relevant?
    precision = tp / k if k > 0 else 0

    # Recall: what % of relevant docs were retrieved?
    recall = tp / len(relevant_set) if len(relevant_set) > 0 else 0

    # F1: harmonic mean balancing precision and recall
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'true_positives': tp,
        'retrieved_count': k,
        'relevant_count': len(relevant_set)
    }

In [43]:
# Evaluate retrieval for all queries
print("\nEvaluating retrieval for all queries...")
retrieval_results = []

for idx, eval_query in enumerate(eval_queries, 1):
    query = eval_query['question']
    relevant_ids = eval_query['relevant_ticket_ids']

    print(f"  [{idx}/{len(eval_queries)}] Processing query...", end='\r')

    try:
        # Retrieve documents
        results = vector_store.similarity_search(query, k=5)
        retrieved_ids = [doc.metadata['ticket_id'] for doc in results]

        # Calculate metrics at k=3
        metrics = calculate_retrieval_metrics(retrieved_ids, relevant_ids, k=3)
        metrics['query_id'] = eval_query['query_id']
        metrics['question'] = query
        metrics['retrieved'] = retrieved_ids[:3]
        metrics['relevant'] = relevant_ids

        retrieval_results.append(metrics)
    except Exception as e:
        print(f"\n  ⚠ Error on query {idx}: {str(e)[:50]}")
        continue

print(f"\n✓ Completed retrieval evaluation for {len(retrieval_results)} queries")


Evaluating retrieval for all queries...

✓ Completed retrieval evaluation for 15 queries


In [44]:
# Aggregate results
print("\n" + "-"*80)
print("RETRIEVAL METRICS (Averaged across all queries)")
print("-"*80)

avg_precision = np.mean([r['precision'] for r in retrieval_results])
avg_recall = np.mean([r['recall'] for r in retrieval_results])
avg_f1 = np.mean([r['f1'] for r in retrieval_results])

print(f"\nPrecision@3: {avg_precision:.4f}")
print(f"  → What % of retrieved documents are actually relevant?")
print(f"  → Target: > 0.80 for production")

print(f"\nRecall@3:    {avg_recall:.4f}")
print(f"  → What % of all relevant documents did we find?")
print(f"  → Target: > 0.70 for production")

print(f"\nF1 Score@3:  {avg_f1:.4f}")
print(f"  → Balanced measure of retrieval quality")
print(f"  → Target: > 0.75 for production")

# Show per-query breakdown
print("\n" + "-"*80)
print("PER-QUERY RETRIEVAL RESULTS (Sample)")
print("-"*80)

for i in range(min(3, len(retrieval_results))):
    result = retrieval_results[i]
    print(f"\n{result['query_id']}: {result['question'][:60]}...")
    print(f"  Relevant:  {result['relevant']}")
    print(f"  Retrieved: {result['retrieved']}")
    print(f"  Precision: {result['precision']:.2f} | Recall: {result['recall']:.2f} | F1: {result['f1']:.2f}")


--------------------------------------------------------------------------------
RETRIEVAL METRICS (Averaged across all queries)
--------------------------------------------------------------------------------

Precision@3: 0.5333
  → What % of retrieved documents are actually relevant?
  → Target: > 0.80 for production

Recall@3:    0.8667
  → What % of all relevant documents did we find?
  → Target: > 0.70 for production

F1 Score@3:  0.6400
  → Balanced measure of retrieval quality
  → Target: > 0.75 for production

--------------------------------------------------------------------------------
PER-QUERY RETRIEVAL RESULTS (Sample)
--------------------------------------------------------------------------------

Q1: How do I fix authentication failures after password reset?...
  Relevant:  ['TICK-001', 'TICK-011', 'TICK-014']
  Retrieved: ['TICK-001', 'TICK-016', 'TICK-011']
  Precision: 0.67 | Recall: 0.67 | F1: 0.67

Q2: What causes database connection timeout errors?...
  Releva